# Recreating NYCHA's analysis

In [1]:
## import libraries
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from datetime import datetime, timedelta

## STEP 1: Download data

NYCHA downloaded data from January 1, 2021 to September 27, 2025 according to the inspection date. 

In [2]:
vio_df = pd.read_csv('../input/Housing_Maintenance_Code_Violations_20251202.csv', dtype = {'BIN':'object',
                                                                                           'BBL':'object'})

In [5]:
vio_df.columns = vio_df.columns.str.lower()

In [6]:
vio_df.violationid.count()

np.int64(4143079)

In [7]:
vio_df.streetname.count()

np.int64(4143079)

In [8]:
vio_df.bin.count()

np.int64(4138941)

And read in the addresses

In [17]:
dev_df = pd.read_csv('../input/coded_files/addresses_updated_120425.csv', dtype = {'bbl':'object',
                                                                                     'bin#':'object'})

In [18]:
dev_df['bin#'].nunique()

714

In [19]:
dev_df.bbl.nunique()

450

## STEP 2: Create unique identifier

In [28]:
## create a new column that combines bbl and bin
dev_df['bbl_bin'] = dev_df['bbl'] + dev_df['bin#']
vio_df['bbl_bin'] = vio_df['bbl'] + vio_df['bin']

In [29]:
## make a list of the unique identifiers found in the RAD development list
dev_identifier_list = dev_df.bbl_bin.unique().tolist()

In [30]:
## now find any matches
match_df = vio_df[vio_df['bbl_bin'].isin(dev_identifier_list)]

In [31]:
match_df.bbl_bin.nunique()

615

In [32]:
match_df.bin.nunique()

615

In [33]:
match_df.violationid.nunique()

20099

In [34]:
drop_na = match_df.dropna(subset = 'ViolationID').reset_index(drop = True)

KeyError: ['ViolationID']

In [20]:
drop_na.BIN.nunique()

555

In [21]:
drop_dupes = drop_na.drop_duplicates(subset = ['ViolationID','NOVID'], keep = "first").reset_index(drop = True)

In [23]:
drop_dupes.BIN.nunique()

555

In [24]:
drop_dupes.ViolationID.nunique()

20369

In [25]:
drop_dupes.columns

Index(['ViolationID', 'BuildingID', 'RegistrationID', 'BoroID', 'Borough',
       'HouseNumber', 'LowHouseNumber', 'HighHouseNumber', 'StreetName',
       'StreetCode', 'Postcode', 'Apartment', 'Story', 'Block', 'Lot', 'Class',
       'InspectionDate', 'ApprovedDate', 'OriginalCertifyByDate',
       'OriginalCorrectByDate', 'NewCertifyByDate', 'NewCorrectByDate',
       'CertifiedDate', 'OrderNumber', 'NOVID', 'NOVDescription',
       'NOVIssuedDate', 'CurrentStatusID', 'CurrentStatus',
       'CurrentStatusDate', 'NovType', 'ViolationStatus', 'RentImpairing',
       'Latitude', 'Longitude', 'CommunityBoard', 'CouncilDistrict',
       'CensusTract', 'BIN', 'BBL', 'NTA'],
      dtype='object')

In [28]:
drop_dupes['HouseNumber'] = drop_dupes['HouseNumber'].str.strip()
drop_dupes['StreetName'] = drop_dupes['StreetName'].str.strip()

In [31]:
drop_dupes['address'] = drop_dupes['HouseNumber'] + ' ' + drop_dupes['StreetName']

In [33]:
drop_dupes.address.nunique()

1476